# Deletar as tabelas caso existam

Serve para limpar o ambiente antes de criar as tabelas

In [0]:
%sql
DROP TABLE IF EXISTS mvp_engenharia_dados.silver.cid

In [0]:
%sql
DROP TABLE IF EXISTS mvp_engenharia_dados.silver.capitulo_cid

In [0]:
%sql
DROP TABLE IF EXISTS mvp_engenharia_dados.silver.estado

In [0]:
%sql
DROP TABLE IF EXISTS mvp_engenharia_dados.silver.municipio

In [0]:
%sql
DROP TABLE IF EXISTS mvp_engenharia_dados.silver.sim

# Criação das tabelas Silver

Esse notebook tem por objetivo ler as tabelas da camada bronze e transformar os dados brutos em dados confiáveis e estruturados para análise. Para isso, ela envolve os tratamentos relacionados à qualidade, consistência, padronização e organização dos dados. Além disso, essa etapa também prepara os relacionamentos necessários entre SIM, CIDs, municípios e estados.


## Tratamento da tabela *CID*

A tabela Bronze do CID foi obtida a partir de uma planilha Excel do Ministério da Previdência Social, que passou por um pré-processamento mínimo para transformá-la em um formato que pudesse ser carregado como tabela no Databricks. Por esse motivo, a tabela Bronze ainda contém muitas informações que não são relevantes para o trabalho, além de problemas relacionados à sua estrutura. Para o tratamento da tabela, serão realizadas as seguintes etapas:

* Leitura da tabela CID da camada Bronze;

  * Visualização da tabela;
  * Análise de campos vazios e duplicidade de linhas;
* Seleção das colunas e análise da nova estrutura;

  * Seleção das colunas com informações relevantes;
  * Análise da tabela filtrada;
  * Criação de novas colunas;
* Separação das informações em duas novas tabelas: *CID* e *CAPITULOS*;
* Salvamento das tabelas na camada Silver.

O texto a seguir descreve as etapas realizadas, apresenta os principais resultados das análises e explica algumas das decisões tomadas durante o tratamento. Os códigos utilizados em cada processo são apresentados posteriormente.

### 1. Leitura dos dados

Primeiramente, é realizada a leitura da tabela *CID* da camada Bronze. Em seguida, são visualizadas a estrutura da tabela, as informações das colunas e a quantidade de registros duplicados. A partir dessa análise, foi possível identificar que existem linhas e colunas com valores nulos. Também foi verificado que não existem linhas duplicadas na tabela.

### 2. Seleção das colunas e análise da nova tabela

A coluna que contém as informações relevantes sobre os CIDs é a coluna `Coluna_2`. Essa coluna contém diferentes informações relacionadas à classificação, como o código do CID, seu nome e o capítulo ao qual ele pertence.

Primeiramente, é selecionada a coluna de interesse (`Coluna_2`) e, em seguida, é realizada uma análise de seus dados. Nessa análise, foram identificadas linhas no início e no final do DataFrame que devem ser excluídas, pois não fazem parte da classificação dos CIDs. Essas linhas correspondem a informações adicionais presentes na base original que não foram removidas durante o pré-processamento. Por esse motivo, elas precisam ser excluídas nesta etapa.

Após a remoção dessas linhas, foi possível observar que a coluna selecionada é composta basicamente por dois tipos de informação:

* linhas que representam os títulos dos capítulos;
* linhas que contêm informações sobre os CIDs.

As linhas referentes aos capítulos, como `"Capítulo I: Algumas doenças infecciosas e parasitárias"`, apresentam a informação do capítulo utilizado para agrupar os CIDs que aparecem nas linhas seguintes. Já as linhas referentes aos CIDs apresentam o código e o nome correspondente. Na estrutura utilizada neste trabalho, os códigos de interesse são identificados pelo padrão de uma letra seguida de dois números. Dessa forma, para estruturar os dados, as informações presentes na coluna original serão separadas em colunas distintas, sendo uma destinada às informações do CID e outra destinada ao capítulo ao qual ele pertence. Após essa separação, serão removidas as linhas que não apresentarem um código de CID válido de acordo com o padrão definido para esta base (Iniciar com uma letra seguido de dois números).

Além da identificação e limpeza dos registros, as informações ainda precisam ser separadas em seus respectivos atributos. Tanto as informações dos CIDs quanto as informações dos capítulos possuem o código e o nome juntos em uma mesma coluna. Por esse motivo, serão realizados os seguintes tratamentos:

* Separar o código do CID do respectivo nome;
* Separar o código do capítulo do respectivo nome;
* Remover o termo `"Capítulo"` do código do capítulo, mantendo apenas sua numeração romana;
* Remover as colunas utilizadas durante o tratamento que não serão necessárias na tabela final.

### 3. Separação da tabela em tabelas *cid* e *capitulos*

Após a limpeza e o tratamento das informações, observa-se que os registros de CID ainda possuem duas informações relacionadas: os dados do próprio CID e os dados do capítulo ao qual ele pertence. Para organizar melhor essas informações, a estrutura será dividida em duas tabelas.  

Essa decisão está relacionada aos princípios de normalização de bancos de dados. Na **Primeira Forma Normal (1FN)**, busca-se, entre outros aspectos, que os atributos contenham valores atômicos e que cada campo represente uma única informação. A separação das entidades também contribui para evitar a repetição desnecessária de informações e facilita a manutenção dos relacionamentos entre os dados.

A primeira tabela será a *capitulos*, contendo:

* código do capítulo;
* nome do capítulo.

A segunda será a *cid*, contendo:

* código do CID;
* nome do CID;
* identificador do capítulo ao qual o CID pertence.

Dessa forma, as informações dos capítulos não precisam ser repetidas em todos os registros de CID. A tabela *CID* passa a se relacionar com a tabela *CAPITULOS* por meio do identificador do capítulo.

Por fim, são realizadas algumas validações na tabela *CID* para verificar a consistência dos dados tratados:

* Verificar se existe algum código com quantidade de caracteres diferente da esperada;
* Verificar se existe algum código com formatação diferente do padrão definido para esta base (letra + dois números);
* Verificar se existe algum `ID_CAPITULO` menor que 1 ou maior que 21.

Essas verificações têm como objetivo identificar possíveis inconsistências após o processo de limpeza e transformação dos dados.

### 4. Salvamento das tabelas na camada silver

Por fim, as tabelas *cid* e *capitulos* foram salvas na camada silver.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# Leitura e análise exploratória dos dados
# ============================================================

# Leitura da tabela Bronze
cid = spark.table("mvp_engenharia_dados.bronze.cid")

# Visualização da tabela
display(cid)

# Características da tabela
cid.printSchema()

print(f"Quantidade de registros: {cid.count()}")

# Verificação de duplicidades
print(f"Quantidade de registros duplicados: {cid.count() - cid.dropDuplicates().count()}")

In [0]:
# ============================================================
# Seleção das colunas e análise da nova tabela
# ============================================================

# Obter apenas a segunda coluna
cid_limpo = cid.select("Coluna_2")

# Visualizar a tabela
print("Antes da limpeza")
display(cid_limpo)

# Criar uma numeração para as linhas
window_ordem = Window.orderBy(F.monotonically_increasing_id())

cid_limpo = cid_limpo.withColumn(
    "_row_num",
    F.row_number().over(window_ordem)
)

# Quantidade de registros
total_linhas = cid_limpo.count()

# Remover a primeira e as últimas 5 linhas
cid_limpo = cid_limpo.filter(
    (F.col("_row_num") > 1) &
    (F.col("_row_num") <= total_linhas - 5)
)

# Remover a coluna auxiliar
cid_limpo = cid_limpo.drop("_row_num")

# Visualizar a tabela
print("depois da limpeza")
display(cid_limpo)

In [0]:
# ============================================================
# Identificação dos capítulos
# ============================================================

# Criar identificador para manter a ordem dos registros
cid_limpo = cid_limpo.withColumn(
    "_ordem",
    F.monotonically_increasing_id()
)

# Criar coluna com o capítulo
cid_limpo = cid_limpo.withColumn(
    "CAPITULO",
    F.when(
        F.col("Coluna_2").startswith("Capítulo"),
        F.col("Coluna_2")
    )
)

# Propagar o capítulo para as linhas seguintes
window_capitulo = (
    Window
    .orderBy("_ordem")
    .rowsBetween(Window.unboundedPreceding, 0)
)

cid_limpo = cid_limpo.withColumn(
    "CAPITULO",
    F.last("CAPITULO", ignorenulls=True).over(window_capitulo)
)

# Manter apenas os registros que possuem código no formato
# Letra + 2 números
cid_limpo = cid_limpo.filter(
    F.col("Coluna_2").rlike(r"^[a-zA-Z]\d{2}")
)

display(cid_limpo)

In [0]:
# ============================================================
# Separação do código e nome
# ============================================================

# Separar código CID e nome CID
cid_limpo = (
    cid_limpo
    .withColumn(
        "CODIGO_CID",
        F.regexp_extract(
            F.col("Coluna_2"),
            r"^([a-zA-Z]\d{2})\s*-\s*",
            1
        )
    )
    .withColumn(
        "NOME_CID",
        F.regexp_extract(
            F.col("Coluna_2"),
            r"^[a-zA-Z]\d{2}\s*-\s*(.*)",
            1
        )
    )
)

# Separar código e nome do capítulo
cid_limpo = (
    cid_limpo
    .withColumn(
        "CODIGO_CAPITULO",
        F.regexp_extract(
            F.col("CAPITULO"),
            r"^Capítulo\s+([^:]+)",
            1
        )
    )
    .withColumn(
        "NOME_CAPITULO",
        F.regexp_extract(
            F.col("CAPITULO"),
            r"^Capítulo\s+[^:]+\s*:\s*(.*)",
            1
        )
    )
)

display(cid_limpo)

In [0]:
# Remover colunas que não fazem parte da estrutura final
cid_limpo = cid_limpo.select(
    "CODIGO_CID",
    "NOME_CID",
    "CODIGO_CAPITULO",
    "NOME_CAPITULO"
)

display(cid_limpo)

In [0]:
# ============================================================
# Tabela CAPITULO
# ============================================================

df_capitulo = (
    cid_limpo
    .select(
        "CODIGO_CAPITULO",
        "NOME_CAPITULO"
    )
    .dropDuplicates()
)

# Criar ID do capítulo
window_id_capitulo = Window.orderBy("CODIGO_CAPITULO")

df_capitulo = (
    df_capitulo
    .withColumn(
        "ID_CAPITULO",
        F.row_number().over(window_id_capitulo)
    )
    .select(
        "ID_CAPITULO",
        "CODIGO_CAPITULO",
        "NOME_CAPITULO"
    )
)

# Garantir código como string
df_capitulo = df_capitulo.withColumn(
    "CODIGO_CAPITULO",
    F.col("CODIGO_CAPITULO").cast("string")
)

display(df_capitulo)

print(f"Quantidade de capítulos: {df_capitulo.count()}")

In [0]:
# ============================================================
# Tabela CID
# ============================================================

df_cid = (
    cid_limpo
    .select(
        "CODIGO_CID",
        "NOME_CID",
        "CODIGO_CAPITULO"
    )
)

# Criar ID da CID
window_id_cid = Window.orderBy("CODIGO_CID")

df_cid = df_cid.withColumn(
    "ID_CID",
    F.row_number().over(window_id_cid)
)

# Relacionar CID com capítulo
df_cid = (
    df_cid
    .join(
        df_capitulo.select(
            "ID_CAPITULO",
            "CODIGO_CAPITULO"
        ),
        on="CODIGO_CAPITULO",
        how="left"
    )
)

# Selecionar somente as colunas finais
df_cid = df_cid.select(
    "ID_CID",
    "CODIGO_CID",
    "NOME_CID",
    "ID_CAPITULO"
)

display(df_cid)

In [0]:
# ============================================================
# Diagnóstico do DataFrame
# ============================================================

print("--- DIAGNÓSTICO DO DATAFRAME ---")

# 1. Códigos com mais de 3 caracteres
codigos_maiores_3 = df_cid.filter(
    F.length("CODIGO_CID") > 3
).count()

print(
    f"1. Códigos > 3 caracteres: {codigos_maiores_3}"
)

# 2. Formato diferente de Letra + 2 números
formato_invalido = df_cid.filter(
    ~F.col("CODIGO_CID").rlike(r"^[a-zA-Z]\d{2}$")
).count()

print(
    f"2. Formato diferente de Letra+2Números: {formato_invalido}"
)

# 3. Capítulo fora de 1-21
capitulo_invalido = df_cid.filter(
    (F.col("ID_CAPITULO") < 1) |
    (F.col("ID_CAPITULO") > 21) |
    F.col("ID_CAPITULO").isNull()
).count()

print(
    f"3. Capítulo fora de 1-21: {capitulo_invalido}"
)

# 4. Duplicidades
duplicidades_cid = (
    df_cid.count() -
    df_cid.dropDuplicates().count()
)

print(
    f"4. Registros duplicados: {duplicidades_cid}"
)

In [0]:
# ============================================================
# Salvamento das tabelas na camada Silver
# ============================================================

df_cid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.silver.cid")

df_capitulo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.silver.capitulo_cid")

In [0]:
#  verificar tabela cid da camada silver
display(spark.table("mvp_engenharia_dados.silver.cid"))

In [0]:
#  verificar tabela capitulo da camada silver
display(spark.table("mvp_engenharia_dados.silver.capitulo_cid"))


## Tratamento da tabela *IBGE*

A tabela Bronze do IBGE foi obtida a partir de uma planilha contendo informações sobre os municípios e estados brasileiros. Para facilitar a utilização dessas informações e estabelecer uma estrutura adequada para os relacionamentos com as demais tabelas do projeto, serão realizados tratamentos e reorganizações na camada Silver. Serão realizadas as seguintes etapas:
* Leitura da tabela IBGE da camada Bronze;
  * Visualização da tabela;
  * Análise de campos vazios e duplicidade de linhas;
* Seleção das colunas e análise da nova estrutura;
  * Seleção das colunas com informações relevantes;
  * Análise da tabela filtrada;
  * Criação de novas colunas;
* Validação das tabelas;
* Separação das informações em duas novas tabelas: *estado* e *municipio*;
* Salvamento das tabelas na camada Silver.

O texto a seguir descreve as etapas realizadas, apresenta os principais resultados das análises e explica algumas das decisões tomadas durante o tratamento. Os códigos utilizados em cada processo são apresentados posteriormente.

### 1. Leitura dos dados

Primeiramente, é realizada a leitura da tabela *ibge* da camada Bronze. Em seguida, são analisadas suas características, incluindo a quantidade de registros, as colunas disponíveis, a existência de valores nulos e a presença de registros duplicados.

A análise inicial identificou que a tabela possui **5.570 registros** e não apresenta campos nulos nem linhas duplicadas. Apesar disso, a ausência de valores nulos e duplicidades não é suficiente para garantir a qualidade das informações. Por esse motivo, também serão realizadas validações relacionadas aos códigos de estados e municípios.

### 2. Seleção das colunas e análise da nova tabela

Primeiramente, são selecionadas apenas as colunas consideradas relevantes para o projeto, contendo os códigos e nomes dos estados e dos municípios. Após a seleção, os nomes das colunas são modificados para seguir um padrão de nomenclatura utilizado no projeto. Também são realizados os tratamentos necessários nos tipos de dados das colunas.

Em seguida, é realizada uma nova análise da estrutura da tabela filtrada, verificando novamente a existência de valores nulos e registros duplicados. A análise confirma que não existem campos nulos nem registros duplicados. Entretanto, algumas validações adicionais são necessárias para verificar a consistência dos códigos territoriais, sendo estas:
* Verificar se o código do município do IBGE possui 7 dígitos;
* Verificar se o código da UF possui 2 dígitos;
* Verificar se o código da UF está dentro dos códigos válidos dos estados brasileiros;
* Verificar se os dois primeiros dígitos do código do município correspondem ao código da UF à qual o município pertence.

Essas verificações têm como objetivo identificar possíveis inconsistências nos códigos antes que os dados sejam utilizados nos relacionamentos com as demais tabelas.

Um ponto que deve ser destacado é que, além do código oficial do município utilizado na base do IBGE, é necessário considerar a forma como os municípios são identificados nas bases de dados de saúde utilizadas no projeto. O código do município presente nas bases de saúde brasileiro possui **6 dígitos**, enquanto o código do município utilizado na referência do IBGE possui **7 dígitos**. Devido a isso, será criado a coluna `CODIGO_MUNICIPIO_SAUDE`, que será formada a partir dos seis primeiros dígitos do código do município do IBGE e que vai estabelecer o relacionamento entre os registros do SIM e a tabela de municípios.

Dessa forma, a tabela de municípios passará a possuir os dois identificadores:

* `CODIGO_MUNICIPIO_IBGE` — código de 7 dígitos utilizado na referência do IBGE;
* `CODIGO_MUNICIPIO_SAUDE` — código de 6 dígitos utilizado para o relacionamento com as bases de saúde.

### 3. Separação das informações em duas novas tabelas: *estado* e *municipio*

A tabela original do IBGE contém informações relacionadas a duas entidades distintas: **estados** e **municípios**. Para organizar melhor essas informações, a estrutura será dividida em duas tabelas. Destaca-se que, essa decisão está relacionada aos princípios de normalização de bancos de dados (**Primeira Forma Normal**), e busca contribuir para evitar a repetição desnecessária de dados e facilitar a criação dos relacionamentos entre as tabelas.

A primeira tabela será a *estado*, contendo:

* identificador da tabela estado;
* código da UF;
* nome do estado.

A segunda será a *MUNICIPIO*, contendo:

* identificador da tabela municipio;
* código do município do IBGE;
* código do município utilizado pelas bases de saúde;
* nome do município;
* identificador do estado ao qual o município pertence.

Dessa forma, as informações dos estados não precisam ser repetidas como atributos completos em todos os registros de municípios. A tabela *municipio* passa a se relacionar com a tabela *estado* por meio do identificador do estado.

### 4. Validação das tabelas

Após a separação das informações, são realizadas novas verificações para assegurar a consistência das tabelas resultantes. Na tabela *estado*, serão verificadas as informações relacionadas aos códigos e nomes dos estados. já na tabela *municipio*, serão verificadas principalmente:

* existência de códigos de município com quantidade incorreta de dígitos;
* correspondência entre o código do município e o código da UF;
* consistência do `CODIGO_MUNICIPIO_SAUDE`;
* existência de valores nulos;
* existência de registros duplicados.

Essas verificações têm como objetivo garantir que as tabelas estejam consistentes antes de serem disponibilizadas para as próximas etapas do projeto.

### 5. Salvamento das tabelas na camada Silver

Após a conclusão da análise exploratória, das validações e dos tratamentos, as duas tabelas resultantes são salvas na camada Silver:



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# Leitura e análise exploratória dos dados
# ============================================================

# Leitura da tabela Bronze
ibge = spark.table("mvp_engenharia_dados.bronze.ibge")

# Visualizar a tabela
display(ibge)

# Visualizar estrutura
ibge.printSchema()

# Visualizar nomes das colunas
print(ibge.columns)

# Quantidade de registros
print(f"Quantidade de registros: {ibge.count()}")

# Número de linhas duplicadas
duplicados = ibge.count() - ibge.dropDuplicates().count()

print(f"Quantidade de registros duplicados: {duplicados}")

In [0]:
# ============================================================
# Seleção das colunas
# ============================================================

ibge_limpo = ibge.select(
    "UF",
    "Nome_UF",
    "Codigo_Municipio_Completo",
    "Nome_Municipio"
)

# Renomear as colunas
ibge_limpo = (
    ibge_limpo
    .withColumnRenamed("UF", "CODIGO_UF")
    .withColumnRenamed("Nome_UF", "NOME_UF")
    .withColumnRenamed(
        "Codigo_Municipio_Completo",
        "CODIGO_MUNICIPIO_IBGE"
    )
    .withColumnRenamed(
        "Nome_Municipio",
        "NOME_MUNICIPIO"
    )
)

display(ibge_limpo)

In [0]:
# ============================================================
# Ajuste dos tipos de dados das colunas de códigos
# ============================================================

ibge_limpo = (
    ibge_limpo
    .withColumn(
        "CODIGO_UF",
        F.col("CODIGO_UF").cast("string")
    )
    .withColumn(
        "CODIGO_MUNICIPIO_IBGE",
        F.col("CODIGO_MUNICIPIO_IBGE").cast("string")
    )
)

display(ibge_limpo)
ibge_limpo.printSchema()

In [0]:
# ============================================================
# DIAGNÓSTICO DE VALIDAÇÃO
# ============================================================

print("--- DIAGNÓSTICO DE VALIDAÇÃO ---")

# 1. Duplicadas por Estado + Município
duplicados = (
    ibge_limpo
    .groupBy(
        "CODIGO_UF",
        "CODIGO_MUNICIPIO_IBGE"
    )
    .count()
    .filter(F.col("count") > 1)
)

qtd_duplicados = duplicados.count()

print(
    f"1. Linhas duplicadas (par Estado-Município): "
    f"{qtd_duplicados}"
)

# 2. Código do município com exatamente 7 dígitos
mun_invalidos_tam = ibge_limpo.filter(
    ~F.col("CODIGO_MUNICIPIO_IBGE").rlike(r"^\d{7}$")
)

print(
    f"2. Municípios com tamanho diferente de 7 dígitos: "
    f"{mun_invalidos_tam.count()}"
)

# 3. Código da UF dentro dos intervalos válidos
uf = F.col("CODIGO_UF").cast("int")

intervalos_validos = (
    ((uf >= 11) & (uf <= 17)) |
    ((uf >= 21) & (uf <= 29)) |
    ((uf >= 31) & (uf <= 35)) |
    ((uf >= 41) & (uf <= 43)) |
    ((uf >= 50) & (uf <= 53))
)

uf_invalidas = ibge_limpo.filter(
    ~intervalos_validos
)

print(
    f"3. Estados fora dos intervalos válidos do IBGE: "
    f"{uf_invalidas.count()}"
)

# 4. Município deve começar com os 2 dígitos da UF
mun_nao_inicia_uf = ibge_limpo.filter(
    F.substring(
        F.col("CODIGO_MUNICIPIO_IBGE"),
        1,
        2
    ) != F.col("CODIGO_UF")
)

print(
    f"4. Municípios que NÃO começam com os 2 dígitos "
    f"da sua UF: {mun_nao_inicia_uf.count()}"
)

In [0]:
# ============================================================
# Gerar coluna de Código do município utilizado na Saúde
# ============================================================

ibge_limpo = ibge_limpo.withColumn(
    "CODIGO_MUNICIPIO_SAUDE",
    F.substring(
        F.col("CODIGO_MUNICIPIO_IBGE"),
        1,
        6
    )
)

display(ibge_limpo)

In [0]:
# ============================================================
# Tabela ESTADO
# ============================================================

# Selecionar as colunas referentes a Estado
df_estado = (
    ibge_limpo
    .select(
        "CODIGO_UF",
        "NOME_UF"
    )
    .dropDuplicates()
)

# Criar ID do Estado
window_estado = Window.orderBy("CODIGO_UF")

df_estado = (
    df_estado
    .withColumn(
        "ID_ESTADO",
        F.row_number().over(window_estado)
    )
    .select(
        "ID_ESTADO",
        "CODIGO_UF",
        "NOME_UF"
    )
)

display(df_estado)

print(
    "Duplicidades CODIGO_UF + NOME_UF:",
    df_estado
    .groupBy("CODIGO_UF", "NOME_UF")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
# ============================================================
# Tabela MUNICIPIO
# ============================================================

# Selecionar as colunas referentes ao município
df_municipio = (
    ibge_limpo
    .select(
        "CODIGO_MUNICIPIO_IBGE",
        "CODIGO_MUNICIPIO_SAUDE",
        "NOME_MUNICIPIO",
        "CODIGO_UF"
    )
)

# Criar ID do município
window_municipio = Window.orderBy(
    "CODIGO_MUNICIPIO_IBGE"
)

df_municipio = df_municipio.withColumn(
    "ID_MUNICIPIO",
    F.row_number().over(window_municipio)
)

# Relacionar município com estado
df_municipio = (
    df_municipio
    .join(
        df_estado.select(
            "ID_ESTADO",
            "CODIGO_UF"
        ),
        on="CODIGO_UF",
        how="left"
    )
)

# Selecionar apenas as colunas finais
df_municipio = df_municipio.select(
    "ID_MUNICIPIO",
    "CODIGO_MUNICIPIO_IBGE",
    "CODIGO_MUNICIPIO_SAUDE",
    "NOME_MUNICIPIO",
    "ID_ESTADO"
)

display(df_municipio)

# Verificar duplicidades de município
duplicados_municipio = (
    df_municipio
    .groupBy(
        "CODIGO_MUNICIPIO_IBGE",
        "NOME_MUNICIPIO"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicidades CODIGO_MUNICIPIO_IBGE + NOME_MUNICIPIO:",
    duplicados_municipio
)

In [0]:
# ============================================================
# Salvamento na camada Silver
# ============================================================

df_municipio.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.silver.municipio")

df_estado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.silver.estado")

In [0]:
# Verificar tabela municipio da camada silver
display(spark.table("mvp_engenharia_dados.silver.municipio"))

# Verificar tabela estado da camada silver
display(spark.table("mvp_engenharia_dados.silver.estado"))

## Tratamento da tabela *SIM*

A tabela do **Sistema de Informação sobre Mortalidade (SIM)** contém uma grande quantidade de informações relacionadas aos registros de óbitos. Como o objetivo deste trabalho é analisar os óbitos ocorridos entre 2020 e 2025 e responder às perguntas definidas para o MVP, não será necessário realizar o tratamento completo de todas as informações disponíveis na base original.

Dessa forma, será realizado um recorte das colunas relevantes para as análises propostas. Em seguida, os dados selecionados serão analisados e tratados na camada Silver, considerando aspectos relacionados à completude, qualidade e consistência dos dados. Para o tratamento da tabela, serão realizadas as seguintes etapas:

- Leitura da tabela bronze
- Filtrar colunas desejadas
- Analise e tratamento dos dados
    - Completude
    - Qualidade
    - Consistencia
    - Criação de colunas
- Salvamento na camada Silver

O texto a seguir descreve as etapas realizadas, apresenta os principais resultados das análises e explica algumas das decisões tomadas durante o tratamento.

### 1. Leitura da tabela bronze

Primeiramente, é realizada a leitura da tabela *sim* da camada Bronze e a visualização das informações disponíveis. Como a base original possui uma quantidade elevada de colunas, a primeira etapa consiste em analisar sua estrutura e identificar quais informações são necessárias para responder às perguntas definidas no projeto. Após essa análise, será realizado o recorte das colunas que serão utilizadas no restante do processamento.

### 2. Seleção das colunas

A tabela do SIM possui muitas informações e seu tratamento completo exigiria um escopo maior do que o necessário para este trabalho. Por esse motivo, serão selecionadas apenas as colunas consideradas relevantes para responder às perguntas do MVP. O tratamento e a análise dos dados serão realizados com base nessas informações.

As colunas selecionadas são:

| Campo        | Descrição                                                    |
| ------------ | ------------------------------------------------------------ |
| `TIPOBITO`   | Tipo do óbito, indicando se é fetal ou não fetal             |
| `DTOBITO`    | Data do óbito                                                |
| `HORAOBITO`  | Hora do óbito                                                |
| `DTNASC`     | Data de nascimento                                           |
| `SEXO`       | Sexo da pessoa que veio a óbito                              |
| `CODMUNRES`  | Código do município de residência da pessoa que veio a óbito |
| `CODMUNOCOR` | Código do município onde ocorreu o óbito                     |
| `CAUSABAS`   | Causa básica do óbito                                        |

Algumas considerações foram definidas para a utilização dessas informações:
* A `HORAOBITO` será mantida principalmente como informação auxiliar na análise de possíveis registros duplicados.
* Embora a base do SIM possua informações relacionadas à idade, neste trabalho a idade será calculada a partir da diferença entre a data de nascimento e a data do óbito. Essa decisão permite utilizar diretamente as duas datas disponíveis para gerar uma variável de idade padronizada para a análise.
* Em relação às causas de óbito, será considerada apenas a **causa básica**, representada pelo campo `CAUSABAS`. As demais informações relacionadas a outras causas não serão utilizadas neste MVP.


### 3. Analise e tratamento dos dados

Após a seleção das colunas, será realizada uma análise exploratória dos dados, considerando principalmente sua completude, qualidade e consistência. Inicialmente, serão verificadas a existência de valores nulos e a presença de registros duplicados. 

Com a identificação das duplicidades, estes registros serão removidos. Destaca-se que a `HORAOBITO` será utilizada como uma das informações auxiliares nessa verificação, juntamente com as demais informações disponíveis no recorte da tabela. 

Também serão realizados os ajustes necessários nos tipos de dados das colunas, garantindo que datas, horários, códigos e demais informações estejam em formatos adequados para as análises posteriores.

#### 3.1 Validação dos códigos de municípios

Os códigos dos municípios presentes nos campos `CODMUNRES` e `CODMUNOCOR` serão analisados para verificar sua consistência. Para isso, será verificado se os códigos possuem **6 dígitos**, considerando o padrão utilizado nos dados de saúde. Essa validação é importante porque os códigos presentes no SIM serão posteriormente relacionados à coluna `CODIGO_MUNICIPIO_SAUDE` criada na tabela *municipio* da camada Silver.

#### 3.2 Tratamento da causa básica do óbito

A coluna `CAUSABAS` contém o código da causa básica do óbito de acordo com a classificação utilizada pelo SIM. Inicialmente, serão removidos registros que não apresentem um código compatível com o padrão esperado para os códigos utilizados neste trabalho. Em seguida, a informação da causa básica será separada em duas partes:
* **CID geral** — correspondente aos três primeiros caracteres do código;
* **CID específico** — correspondente ao código completo disponível no registro.

A distinção será utilizada para permitir diferentes níveis de análise das causas de óbito.

**Observações sobre os códigos CID: **

* A classificação CID-10 utiliza códigos alfanuméricos e, em determinadas situações, pode utilizar um ponto após os três primeiros caracteres para representar uma subdivisão ou subcategoria.

* Na base do SIM utilizada neste trabalho, os códigos são disponibilizados sem o ponto. Dessa forma, a estrutura observada no campo `CAUSABAS` será utilizada para separar o código geral e o código específico conforme o padrão presente na própria base.

* Também existem códigos da CID que utilizam a letra `X` como parte de sua estrutura. Portanto, a presença dessa letra não deve ser tratada automaticamente como erro ou como ausência de definição. Ela faz parte da própria codificação de determinadas categorias e subcategorias da classificação.

O tratamento realizado neste projeto deverá preservar os códigos válidos e apenas remover ou classificar registros que não apresentem uma estrutura compatível com o padrão definido para a análise.

#### 3.3. Validação das datas

Outra verificação importante será realizada entre as datas de nascimento e de óbito. Será identificado se existem registros em que a **data do óbito seja anterior à data de nascimento**. Quando esse tipo de inconsistência for identificado, o registro será excluído da base utilizada no projeto.

Essa decisão representa uma regra de tratamento definida especificamente para este trabalho. Quando a data de óbito é anterior à data de nascimento, não é possível determinar, apenas com essas informações, qual das duas datas está incorreta. Dessa forma, o registro não apresenta confiabilidade suficiente para o cálculo da idade e será removido das análises.

Destaca-se que, registros sem data de nascimento não serão necessariamente excluídos. Nesses casos, a idade será considerada como não informada.

#### 3.4. Criação e tratamento de novas colunas

Após os tratamentos iniciais, serão criadas novas colunas para facilitar as análises posteriores.
* Será criada a coluna `CODESTRES`, que representa o código da unidade federativa correspondente ao município de residência da pessoa que veio a óbito. Obtida a partir do código do município onde a pessoa residia.
* Será criada a coluna `COD_UF_OCOR`, que representa o código da unidade federativa correspondente ao município de ocorrência do óbito. Obtida a partir do código do município onde ocorreu o óbito.
* A coluna `SEXO` será analisada para verificar os valores existentes e identificar possíveis inconsistências. Os valores nulos serão tratados como **"Não informado"**, permitindo que esses registros sejam mantidos nas análises sem que a ausência da informação seja confundida com uma categoria de sexo existente.
* Será criada a coluna `ANO_OBITO` a partir da coluna `DTOBITO`. Essa coluna representa o ano em que ocorreu o óbito e será utilizada principalmente nas análises temporais, permitindo observar a evolução dos registros entre 2020 e 2025.
* A idade será calculada a partir da diferença entre a data de óbito e a data de nascimento. Quando a data de nascimento não estiver disponível, a idade será considerada nula.
* A partir da idade calculada será criada a coluna `FAIXA_ETARIA`. Sua distribuição será:

| Faixa etária            | Critério                  |
| ----------------------- | ------------------------- |
| Menores de 1 ano        | idade < 1                 |
| De 1 ano até 17 anos    | idade >= 1 e idade <= 17  |
| De 18 anos até 59 anos  | idade >= 18 e idade <= 59 |
| De 60 anos até 79 anos  | idade >= 60 e idade <= 79 |
| 80 anos ou mais         | idade >= 80               |
| Sem informação de idade | idade nula                |

**OBS:** A categoria **"Sem informação de idade"** será utilizada para os registros em que não for possível calcular a idade devido à ausência da data de nascimento.

### 4. Salvamento da tabela na camada Silver

Após a conclusão da análise, dos tratamentos e das validações, a tabela resultante será salva na camada Silver com o nome `sim`.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# Leitura do banco
# ============================================================

# Leitura da tabela Bronze
sim = spark.table("mvp_engenharia_dados.bronze.sim")

# Visualização das primeiras linhas
display(sim.limit(10))

# Visualização da estrutura da tabela
sim.printSchema()

# Visualização das colunas
print(sim.columns)

# Quantidade de registros
print(f"Quantidade de registros: {sim.count()}")

In [0]:
# ============================================================
# Filtrar colunas desejadas
# ============================================================

sim_limpo = sim.select(
    "TIPOBITO",
    "DTOBITO",
    "HORAOBITO",
    "DTNASC",
    "SEXO",
    "CODMUNRES",
    "CODMUNOCOR",
    "CAUSABAS"
)

# Visualização
display(sim_limpo.limit(10))

# Estrutura
sim_limpo.printSchema()

In [0]:
# ============================================================
# Análise da completude dos dados
# ============================================================

print("--- ANÁLISE DE COMPLETUDE ---")

# Quantidade de nulos por coluna
for coluna in sim_limpo.columns:
    qtd_nulos = sim_limpo.filter(
        F.col(coluna).isNull()
    ).count()

    print(f"{coluna}: {qtd_nulos} nulos")

In [0]:
# ============================================================
# Análise das duplicidades
# ============================================================

# número de linhas antes da remoção de duplicidades
total_antes = sim_limpo.count()

# número de linhas sem duplicidades
total_sem_duplicados = sim_limpo.dropDuplicates().count()

# total das linhas duplicadas
total_duplicados = total_antes - total_sem_duplicados
print(
    f"Total de linhas duplicadas: {total_duplicados}")

# Remover registros duplicados
sim_limpo = sim_limpo.dropDuplicates()
print(
    f"Quantidade de registros após remoção de duplicidades: "
    f"{sim_limpo.count()}"
)

In [0]:
# ============================================================
# Ajuste no formato das datas (assegurar formato date)
# ============================================================

sim_limpo = (
    sim_limpo
    .withColumn(
        "DTOBITO",
        F.expr("try_to_date(DTOBITO, 'yyyy-MM-dd')")
    )
    .withColumn(
        "DTNASC",
        F.expr("try_to_date(DTNASC, 'yyyy-MM-dd')")
    )
)


In [0]:
# ============================================================
# Tratamento da hora para o formato hh:mm
# ============================================================

sim_limpo = sim_limpo.withColumn(
    "HORAOBITO",
    F.when(
        F.col("HORAOBITO").isNull() |
        (F.upper(F.trim(F.col("HORAOBITO"))) == "NA"),
        None
    ).otherwise(
        F.concat(
            F.substring(
                F.lpad(
                    F.expr("try_cast(HORAOBITO as int)").cast("string"),
                    4,
                    "0"
                ),
                1,
                2
            ),
            F.lit(":"),
            F.substring(
                F.lpad(
                    F.expr("try_cast(HORAOBITO as int)").cast("string"),
                    4,
                    "0"
                ),
                3,
                2
            )
        )
    )
)

In [0]:
# ============================================================
# Assegurar o formato dos códigos dos municípios como strings
# ============================================================

sim_limpo = (
    sim_limpo
    .withColumn(
        "CODMUNRES",
        F.col("CODMUNRES").cast("string")
    )
    .withColumn(
        "CODMUNOCOR",
        F.col("CODMUNOCOR").cast("string")
    )
)

In [0]:
# ============================================================
# Validação dos códigos dos municípios (verificar se existe algum código com mais de 6 dígitos)
# ============================================================

munres_longos = sim_limpo.filter(
    F.length(F.trim(F.col("CODMUNRES"))) > 6
)

munocor_longos = sim_limpo.filter(
    F.length(F.trim(F.col("CODMUNOCOR"))) > 6
)

print(
    f"CODMUNRES com mais de 6 dígitos: "
    f"{munres_longos.count()}"
)

print(
    f"CODMUNOCOR com mais de 6 dígitos: "
    f"{munocor_longos.count()}"
)

In [0]:
# ============================================================
# Validação da CAUSABAS (Verificar se todos os campos começam com letra seguida de dois números)
# ============================================================

# Filtrar linhas com formato de CAUSABAS inválido
causas_invalidas = sim_limpo.filter(
    ~F.upper(
        F.trim(F.col("CAUSABAS"))
    ).rlike(r"^[A-Z]\d{2}")
)

print(
    "Formato diferente de Letra+2Números:",
    causas_invalidas.count()
)

# Manter apenas causas básicas válidas
sim_limpo = sim_limpo.filter(
    F.upper(
        F.trim(F.col("CAUSABAS"))
    ).rlike(r"^[A-Z]\d{2}")
)

In [0]:
# ============================================================
# Separação da causa básica em CAUSABAS_GERAL e CAUSABAS_ESPECIFICO
# ============================================================

sim_limpo = (
    sim_limpo
    .withColumn(
        "CAUSABAS_GERAL",
        F.substring(
            F.upper(F.trim(F.col("CAUSABAS"))),
            1,
            3
        )
    )
    .withColumn(
        "CAUSABAS_ESPECIFICO",
        F.substring(
            F.upper(F.trim(F.col("CAUSABAS"))),
            4,
            10
        )
    )
)

# Remover coluna CAUSABAS
sim_limpo = sim_limpo.drop("CAUSABAS")

In [0]:
# ============================================================
# Tratamento dos registros nulos da coluna sexo
# ============================================================

# Substituir valores nulos por Não informado
sim_limpo = sim_limpo.withColumn(
    "SEXO",
    F.when(
        F.col("SEXO").isNull(),
        "Não informado"
    ).otherwise(
        F.col("SEXO")
    )
)
display(
    sim_limpo
    .groupBy("SEXO")
    .agg(
        F.count("*").alias("QNT_REGISTROS")
    )
    .orderBy("SEXO")
)

In [0]:
# ============================================================
# Validação das datas
# ============================================================

# verificar linhas com data de nascimento maior que data de óbito
obito_invalido = sim_limpo.filter(
    F.col("DTOBITO") < F.col("DTNASC")
)

print(
    "Total de registros com data de óbito "
    "anterior ao nascimento:",
    obito_invalido.count()
)

display(
    obito_invalido.select(
        "DTNASC",
        "DTOBITO"
    )
)

# Quantidade antes da exclusão
quantidade_antes = sim_limpo.count()

print(
    f"Quantidade de registros antes da exclusão: "
    f"{quantidade_antes}"
)

# Remover registros com datas inconsistentes (DTOBITO < DTNASC)
sim_limpo = sim_limpo.filter(
    ~(
        F.col("DTOBITO") < F.col("DTNASC")
    )
)

# Quantidade depois
quantidade_depois = sim_limpo.count()

print(
    f"Quantidade de registros depois da exclusão: "
    f"{quantidade_depois}"
)

print(
    f"Registros excluídos: "
    f"{quantidade_antes - quantidade_depois}"
)

In [0]:
# ============================================================
# Cálculo da idade (DTOBITO - DTNASC)
# ============================================================

sim_limpo = sim_limpo.withColumn(
    "IDADE",
    F.when(
        F.col("DTNASC").isNull(),
        None
    ).otherwise(
        F.floor(
            F.months_between(
                F.col("DTOBITO"),
                F.col("DTNASC")
            ) / 12
        )
    )
)

In [0]:
# ============================================================
# Criar coluna de faixa etária a partir da idade
# ============================================================

sim_limpo = sim_limpo.withColumn(
    "FAIXA_ETARIA",
    F.when(
        F.col("IDADE").isNull(),
        "Sem informação de idade"
    )
    .when(
        F.col("IDADE") < 1,
        "menores de 1 ano"
    )
    .when(
        (F.col("IDADE") >= 1) &
        (F.col("IDADE") <= 17),
        "de 1 ano até 17 anos"
    )
    .when(
        (F.col("IDADE") >= 18) &
        (F.col("IDADE") <= 59),
        "de 18 anos até 59 anos"
    )
    .when(
        (F.col("IDADE") >= 60) &
        (F.col("IDADE") <= 79),
        "de 60 anos até 79 anos"
    )
    .when(
        F.col("IDADE") >= 80,
        "maiores de 80 anos"
    )
)

display(
    sim_limpo
    .groupBy("FAIXA_ETARIA")
    .count()
    .orderBy("FAIXA_ETARIA")
)

In [0]:
# ============================================================
# Criar a coluna referente ao ano do óbito
# ============================================================

sim_limpo = sim_limpo.withColumn(
    "ANO_OBITO",
    F.year(F.col("DTOBITO"))
)

In [0]:
# ============================================================
# Criar colunas referentes aos códigos dos estados (residência e óbito)
# ============================================================

sim_limpo = (
    sim_limpo
    .withColumn(
        "CODESTRES",
        F.substring(
            F.trim(F.col("CODMUNRES")),
            1,
            2
        )
    )
    .withColumn(
        "CODESTOCOR",
        F.substring(
            F.trim(F.col("CODMUNOCOR")),
            1,
            2
        )
    )
)

In [0]:
# ============================================================
# Criar ID da tabela de óbito
# ============================================================

window_id = Window.orderBy(
    "DTOBITO",
    "CODMUNRES",
    "CODMUNOCOR"
)

sim_limpo = sim_limpo.withColumn(
    "IDOBITO",
    F.row_number().over(window_id)
)

In [0]:
# ============================================================
# Reordenar colunas
# ============================================================

sim_limpo = sim_limpo.select(
    "IDOBITO",
    "TIPOBITO",
    "DTOBITO",
    "HORAOBITO",
    "ANO_OBITO",
    "DTNASC",
    "IDADE",
    "FAIXA_ETARIA",
    "SEXO",
    "CODMUNRES",
    "CODESTRES",
    "CODMUNOCOR",
    "CODESTOCOR",
    "CAUSABAS_GERAL",
    "CAUSABAS_ESPECIFICO"
)

display(sim_limpo)
sim_limpo.printSchema()

In [0]:
# ============================================================
# Salvamento na camada Silver
# ============================================================

sim_limpo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp_engenharia_dados.silver.sim")

In [0]:
# verificar tabela sim da camada silver
display(
    spark.table("mvp_engenharia_dados.silver.sim")
)